<a href="https://colab.research.google.com/github/willy410-hub/generative-multi-agent-simulation/blob/main/Multi_Agent_Simulation_Groq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install termcolor > /dev/null
!pip install langchain langchain-community langchain-experimental langchain-classic
!pip install faiss-cpu
!pip install langchain-groq langchain-huggingface sentence-transformers

from datetime import datetime, timedelta
from typing import List
import math
import faiss
import os
import logging
logging.basicConfig(level=logging.ERROR)

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.docstore import InMemoryDocstore
from langchain_community.vectorstores import FAISS

from langchain_classic.retrievers import TimeWeightedVectorStoreRetriever

from termcolor import colored
from langchain_experimental.generative_agents import (
    GenerativeAgent,
    GenerativeAgentMemory,
)

In [122]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [ ]:
USER_NAME = "Walid"  # The name you want to use when interviewing the agent.

LLM = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.7)

## Implementing Your First Generative Agent




In [ ]:


def relevance_score_fn(score: float) -> float:
    """Return a similarity score on a scale [0, 1]."""
    return 1.0 - score / math.sqrt(2)

def create_new_memory_retriever():
    """Create a new vector store retriever unique to the agent."""

    embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    embedding_size = 384
    index = faiss.IndexFlatL2(embedding_size)
    vectorstore = FAISS(
        embeddings_model.embed_query,
        index,
        InMemoryDocstore({}),
        {},
        relevance_score_fn=relevance_score_fn,
    )
    return TimeWeightedVectorStoreRetriever(
        vectorstore=vectorstore, other_score_keys=["importance"], k=15
    )

In [ ]:
Gem_memory = GenerativeAgentMemory(
    llm=LLM,
    memory_retriever=create_new_memory_retriever(),
    verbose=False,
    reflection_threshold=8,
)

# Defining the Generative Agent: Gem
Gem = GenerativeAgent(
    name="Gem",
    age=28,
    traits="curious, creative writer, world traveler",
    status="exploring the intersection of technology and storytelling",
    memory_retriever=create_new_memory_retriever(),
    llm=LLM,
    memory=Gem_memory,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# The current "Summary" of a character can't be made because the agent hasn't made
# any observations yet.
print(Gem.get_summary())

Name: Gem (age: 28)
Innate traits: curious, creative writer, world traveler
I don't have information about Gem. Could you please provide more context or information about Gem so I can summarize its core characteristics accurately?


In [ ]:
# We can add memories directly to the memory object

Gem_observations = [
    "Gem recalls her morning walk in the park",
    "Gem feels excited about the new book she started reading",
    "Gem remembers her conversation with a close friend",
    "Gem thinks about the painting she saw at the art gallery",
    "Gem is planning to learn a new recipe for dinner",
    "Gem is looking forward to her weekend trip",
    "Gem contemplates her goals for the month."
]

for observation in Gem_observations:
    Gem.memory.add_memory(observation)



# We will see how this summary updates after more observations to create a more rich description.
print(Gem.get_summary(force_refresh=True))

Name: Gem (age: 28)
Innate traits: curious, creative writer, world traveler
Based on the given statements, Gem's core characteristics can be summarized as:

1. Productive: Gem plans to learn a new recipe, contemplates her goals, and is looking forward to her weekend trip, indicating a proactive and goal-oriented personality.
2. Reflective: Gem contemplates her goals, remembers her conversation with a close friend, and recalls her morning walk in the park, showing a reflective and introspective nature.
3. Curious: Gem feels excited about the new book she started reading and thinks about the painting she saw at the art gallery, indicating an interest in learning and exploring new things.
4. Optimistic: Gem feels excited about the new book and is looking forward to her weekend trip, suggesting a positive and enthusiastic outlook on life.


## Interacting and Providing Context to Generative Characters

## Pre-Interview with Character

Before sending our character on their way, let's ask them a few questions.

In [ ]:
def interview_agent(agent: GenerativeAgent, message: str) -> str:
    """Help the notebook user interact with the agent."""
    new_message = f"{USER_NAME} says {message}"
    return agent.generate_dialogue_response(new_message)[1]

In [ ]:
interview_agent(Gem, "What do you like to do?")


/usr/local/lib/python3.12/dist-packages/langchain_classic/retrievers/time_weighted_retriever.py:86: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='02ba3fec-6183-40db-871d-3bf48de395b7', metadata={'importance': 0.105, 'last_accessed_at': datetime.datetime(2026, 7, 30, 15, 40, 40, 937787), 'created_at': datetime.datetime(2026, 7, 30, 15, 40, 40, 937787), 'buffer_idx': 2}, page_content='Gem remembers her conversation with a close friend'), np.float32(-0.11582136)), (Document(id='663ba359-3c82-48f7-a233-525b793b2b06', metadata={'importance': 0.06, 'last_accessed_at': datetime.datetime(2026, 7, 30, 15, 40, 40, 249975), 'created_at': datetime.datetime(2026, 7, 30, 15, 40, 40, 249975), 'buffer_idx': 0}, page_content='Gem recalls her morning walk in the park'), np.float32(-0.14380002)), (Document(id='3587e463-24e3-42ec-8ab1-ce1019bb7a20', metadata={'importance': 0.03, 'last_accessed_at': datetime.datetime(2026, 7, 30, 15, 40, 41, 123592), 'created_at': datetime.datet

'Gem said "I\'m really passionate about exploring the intersection of technology and storytelling, so I\'ve been experimenting with digital art and writing interactive stories. I\'m also a world traveler, and I love meeting new people and hearing their stories, which often inspires my writing."'

## Step through the day's observations.

In [ ]:
# Let's give Gem a series of observations to reflect on her day
# Adding observations to Gem' memory
Gem_observations_day = [
    "Gem starts her day with a refreshing yoga session.",
    "Gem spends time writing in her journal.",
    "Gem experiments with a new recipe she found online.",
    "Gem gets lost in her thoughts while gardening.",
    "Gem decides to call her grandmother for a heartfelt chat.",
    "Gem relaxes in the evening by playing her favorite piano pieces.",
]

for observation in Gem_observations_day:
    Gem.memory.add_memory(observation)


In [ ]:
# Let's observe how Gem's day influences her memory and character
for i, observation in enumerate(Gem_observations_day):
    _,reaction = Gem.generate_reaction(observation)
    print(colored(observation, "Red"), reaction)
    if ((i + 1) % len(Gem_observations_day)) == 0:
        print("*" * 40)
        print(
            colored(
                f"After these observations, Gem's summary is:\n{Gem.get_summary(force_refresh=True)}",
                "blue",
            )
        )
        print("*" * 40)


Gem starts her day with a refreshing yoga session. Gem smiles to herself as she remembers the invigorating yoga session, feeling grateful for the opportunity to start her day with a healthy and centered routine.


/usr/local/lib/python3.12/dist-packages/langchain_classic/retrievers/time_weighted_retriever.py:86: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='b16b5265-ce2b-47f9-84a0-b8516b978507', metadata={'importance': 0.06, 'last_accessed_at': datetime.datetime(2026, 7, 30, 15, 40, 53, 44874), 'created_at': datetime.datetime(2026, 7, 30, 15, 40, 53, 44874), 'buffer_idx': 9}, page_content='Gem spends time writing in her journal.'), np.float32(0.46999162)), (Document(id='7f604684-3aba-46fb-98b0-4c0c84b14930', metadata={'importance': 0.045, 'last_accessed_at': datetime.datetime(2026, 7, 30, 15, 41, 14, 546694), 'created_at': datetime.datetime(2026, 7, 30, 15, 41, 14, 546694), 'buffer_idx': 15}, page_content='Gem observed Gem spends time writing in her journal. and reacted by REACT: Gem smiles to herself as she continues writing in her journal, lost in thought, inspired by the creative process.'), np.float32(0.39077204)), (Document(id='2ce2a96d-6096-481b-8dcc-322e799d7e6

Gem spends time writing in her journal. Gem smiles to herself as she continues writing in her journal, lost in thought, inspired by the creative process.
Gem experiments with a new recipe she found online. Gem smiles to herself as she continues planning the new recipe, her creative mind already thinking about the flavors and presentation.
Gem gets lost in her thoughts while gardening. Gem smiles to herself, enjoying the sense of connection she feels with nature as she gets lost in thought while gardening.


/usr/local/lib/python3.12/dist-packages/langchain_classic/retrievers/time_weighted_retriever.py:86: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='6c378ac9-3a23-4f06-8406-102a29d84b58', metadata={'importance': 0.06, 'last_accessed_at': datetime.datetime(2026, 7, 30, 15, 42, 32, 852597), 'created_at': datetime.datetime(2026, 7, 30, 15, 42, 32, 852597), 'buffer_idx': 18}, page_content='Gem observed Gem decides to call her grandmother for a heartfelt chat. and reacted by REACT: Gem smiles to herself, feeling a sense of connection and love for her grandmother as she decides to call her for a heartfelt chat.'), np.float32(0.5068311)), (Document(id='686b4f3e-3930-443c-b1f7-0b55dc8f246b', metadata={'importance': 0.12, 'last_accessed_at': datetime.datetime(2026, 7, 30, 15, 40, 53, 901312), 'created_at': datetime.datetime(2026, 7, 30, 15, 40, 53, 901312), 'buffer_idx': 12}, page_content='Gem decides to call her grandmother for a heartfelt chat.'), np.float32(0.4178258

Gem decides to call her grandmother for a heartfelt chat. Gem smiles to herself, feeling a sense of connection and love for her grandmother as she decides to call her for a heartfelt chat.
Gem relaxes in the evening by playing her favorite piano pieces. Gem smiles to herself as she continues playing her favorite piano pieces, feeling a sense of calm and relaxation wash over her.
****************************************
After these observations, Gem's summary is:
Name: Gem (age: 28)
Innate traits: curious, creative writer, world traveler
Gem's core characteristics can be summarized as:

1. **Creative and thoughtful**: Gem enjoys trying new recipes, thinking about flavors and presentation, and writing in her journal, showing her creative side.
2. **Reflective and contemplative**: Gem spends time thinking about her goals, conversations with friends, and memories, indicating a reflective and contemplative nature.
3. **Grateful and appreciative**: Gem feels grateful for her yoga practice, c

In [ ]:
def interview_agent(agent: GenerativeAgent, message: str) -> str:
    """Help the notebook user interact with the agent."""
    new_message = f"{USER_NAME} says {message}"
    return agent.generate_dialogue_response(new_message)[1]

In [ ]:
interview_agent(Gem, "What do you like to do?")

/usr/local/lib/python3.12/dist-packages/langchain_classic/retrievers/time_weighted_retriever.py:86: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='aca54c7e-dbcc-49eb-a458-865fc389b5e6', metadata={'importance': 0.12, 'last_accessed_at': datetime.datetime(2026, 7, 30, 15, 40, 50, 941151), 'created_at': datetime.datetime(2026, 7, 30, 15, 40, 50, 941151), 'buffer_idx': 7}, page_content='Gem observed Walid says What do you like to do? and said "I\'m really passionate about exploring the intersection of technology and storytelling, so I\'ve been experimenting with digital art and writing interactive stories. I\'m also a world traveler, and I love meeting new people and hearing their stories, which often inspires my writing."'), np.float32(0.24582726)), (Document(id='6c378ac9-3a23-4f06-8406-102a29d84b58', metadata={'importance': 0.06, 'last_accessed_at': datetime.datetime(2026, 7, 30, 15, 42, 32, 852597), 'created_at': datetime.datetime(2026, 7, 30, 15, 42, 32, 8525

'Gem said "I\'m really passionate about exploring the intersection of technology and storytelling, so I\'ve been experimenting with digital art and writing interactive stories. I\'m also a world traveler, and I love meeting new people and hearing their stories, which often inspires my writing."'

In [ ]:
# Let's give Gem a series of observations to reflect on her day
# Adding observations to Gem' memory
alexis_observations_day = [
    "Gem starts her day with a refreshing yoga session.",
    "Gem spends time writing in her journal.",
    "Gem experiments with a new recipe she found online.",
    "Gem gets lost in her thoughts while gardening.",
    "Gem decides to call her grandmother for a heartfelt chat.",
    "Gem relaxes in the evening by playing her favorite piano pieces.",
]

for observation in Gem_observations_day:
    Gem.memory.add_memory(observation)

In [ ]:
# Let's observe how Gem's day influences her memory and character
for i, observation in enumerate(Gem_observations_day):
    _, reaction = Gem.generate_reaction(observation)
    print(colored(observation, "Red"), reaction)
    if ((i + 1) % len(Gem_observations_day)) == 0:
        print("*" * 40)
        print(
            colored(
                f"After these observations, Gem's summary is:\n{Gem.get_summary(force_refresh=True)}",
                "blue",
            )
        )
        print("*" * 40)

Gem starts her day with a refreshing yoga session. Gem smiles to herself as she remembers the invigorating yoga session, feeling grateful for the opportunity to start her day with a healthy and centered routine.
Gem spends time writing in her journal. Gem smiles to herself as she continues writing in her journal, lost in thought, inspired by the creative process.


/usr/local/lib/python3.12/dist-packages/langchain_classic/retrievers/time_weighted_retriever.py:86: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='081278bc-ba26-4408-afee-618a061b1091', metadata={'importance': 0.045, 'last_accessed_at': datetime.datetime(2026, 7, 30, 15, 40, 53, 325942), 'created_at': datetime.datetime(2026, 7, 30, 15, 40, 53, 325942), 'buffer_idx': 10}, page_content='Gem experiments with a new recipe she found online.'), np.float32(0.46590638)), (Document(id='0682b981-cde6-4524-88f8-2fdf82f8c7e3', metadata={'importance': 0.03, 'last_accessed_at': datetime.datetime(2026, 7, 30, 15, 46, 51, 317887), 'created_at': datetime.datetime(2026, 7, 30, 15, 46, 51, 317887), 'buffer_idx': 28}, page_content='Gem experiments with a new recipe she found online.'), np.float32(0.46590638)), (Document(id='c84f72b6-63d8-4382-a608-ff1ba6e6015c', metadata={'importance': 0.09, 'last_accessed_at': datetime.datetime(2026, 7, 30, 15, 41, 40, 87412), 'created_at': dat

Gem experiments with a new recipe she found online. Gem smiles to herself as she continues planning the new recipe, her creative mind already thinking about the flavors and presentation.


/usr/local/lib/python3.12/dist-packages/langchain_classic/retrievers/time_weighted_retriever.py:86: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='2ce2a96d-6096-481b-8dcc-322e799d7e6b', metadata={'importance': 0.045, 'last_accessed_at': datetime.datetime(2026, 7, 30, 15, 42, 4, 425503), 'created_at': datetime.datetime(2026, 7, 30, 15, 42, 4, 425503), 'buffer_idx': 17}, page_content='Gem observed Gem gets lost in her thoughts while gardening. and reacted by REACT: Gem smiles to herself, enjoying the sense of connection she feels with nature as she gets lost in thought while gardening.'), np.float32(0.42766905)), (Document(id='ab252c39-44dd-4dc8-ac98-e08167e3d6e4', metadata={'importance': 0.06, 'last_accessed_at': datetime.datetime(2026, 7, 30, 15, 44, 51, 235857), 'created_at': datetime.datetime(2026, 7, 30, 15, 44, 51, 235857), 'buffer_idx': 22}, page_content='Gem observed Gem gets lost in her thoughts while gardening. and reacted by REACT: Gem smiles to hers

Gem gets lost in her thoughts while gardening. Gem smiles to herself, enjoying the sense of connection she feels with nature as she gets lost in thought while gardening.


/usr/local/lib/python3.12/dist-packages/langchain_classic/retrievers/time_weighted_retriever.py:86: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='6c378ac9-3a23-4f06-8406-102a29d84b58', metadata={'importance': 0.06, 'last_accessed_at': datetime.datetime(2026, 7, 30, 15, 42, 32, 852597), 'created_at': datetime.datetime(2026, 7, 30, 15, 42, 32, 852597), 'buffer_idx': 18}, page_content='Gem observed Gem decides to call her grandmother for a heartfelt chat. and reacted by REACT: Gem smiles to herself, feeling a sense of connection and love for her grandmother as she decides to call her for a heartfelt chat.'), np.float32(0.4175011)), (Document(id='9bad013e-c8ec-4d65-be42-2a51acf4e092', metadata={'importance': 0.075, 'last_accessed_at': datetime.datetime(2026, 7, 30, 15, 45, 25, 718472), 'created_at': datetime.datetime(2026, 7, 30, 15, 45, 25, 718472), 'buffer_idx': 23}, page_content='Gem observed Gem decides to call her grandmother for a heartfelt chat. and react

Gem decides to call her grandmother for a heartfelt chat. Gem smiles to herself, feeling a sense of connection and love for her grandmother as she decides to call her for a heartfelt chat.
Gem relaxes in the evening by playing her favorite piano pieces. Gem smiles to herself as she continues playing her favorite piano pieces, feeling a sense of calm and relaxation wash over her.
****************************************
After these observations, Gem's summary is:
Name: Gem (age: 28)
Innate traits: curious, creative writer, world traveler
Gem's core characteristics appear to be:

1. Creative: Gem is often observed experimenting with new recipes or writing in her journal, indicating her creative mind and interest in self-expression.
2. Reflective: Gem gets lost in thought while gardening or writing, showing that she values introspection and contemplation.
3. Positive: Gem's reactions are consistently described as smiling to herself, indicating a generally optimistic and happy demeanor.
4.

# DialogueAgent and DialogueSimulator **Classes**
1. The DialogueAgent Class.

    
    Responsible for managing an individual agent's system prompt, message history, and LLM invocation.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq

class DialogueAgent:
    def __init__(
        self,
        name: str,
        system_message: SystemMessage,
        model: ChatGroq,
    ) -> None:
        self.name = name
        self.system_message = system_message
        self.model = model
        self.prefix = f"{self.name}: "
        self.reset()

    def reset(self):
        self.message_history = ["Here is the conversation so far."]

    def send(self) -> str:
        """
        Applies the chatmodel to the message history
        and returns the message string
        """
        message = self.model.invoke(
            [
                self.system_message,
                HumanMessage(content="\n".join(self.message_history + [self.prefix])),
            ]
        )
        return message.content

    def receive(self, name: str, message: str) -> None:
        """
        Concatenates {message} spoken by {name} into message history
        """
        self.message_history.append(f"{name}: {message}")

2. The DialogueSimulator Class.

    Orchestrates the conversation flow between multiple agents and control

In [ ]:
from typing import List, Callable

class DialogueSimulator:
    def __init__(
        self,
        agents: List[DialogueAgent],
        selection_function: Callable[[int, List[DialogueAgent]], int],
    ) -> None:
        self.agents = agents
        self._step = 0
        self.select_next_speaker = selection_function

    def reset(self):
        for agent in self.agents:
            agent.reset()

    def inject(self, name: str, message: str):
        """
        Initiates the conversation with a {message} from {name}
        """
        for agent in self.agents:
            agent.receive(name, message)

        # increment time
        self._step += 1

    def step(self) -> tuple[str, str]:
        # 1. choose the next speaker
        speaker_idx = self.select_next_speaker(self._step, self.agents)
        speaker = self.agents[speaker_idx]

        # 2. next speaker sends message
        message = speaker.send()

        # 3. everyone receives message
        for receiver in self.agents:
            receiver.receive(speaker.name, message)

        # 4. increment time
        self._step += 1

        return speaker.name, message